# Baseline Modeling

This notebook establishes baseline performance for accident severity prediction using the feature-engineering pipeline defined in `src/features.py`.

Model development is performed using expanding-window temporal cross-validation over the 2020-2023 development period.

The primary evaluation metric is Macro F1, which gives equal importance to the Fatal, Serious, and Slight classes despite their different frequencies. Class-specific performance is also monitored, with particular attention to the Fatal class.

All preprocessing steps are fitted independently within each training fold to prevent information leakage.

The 2024 final holdout set remains sealed and is not loaded or inspected during model development.

## Setup

### Imports

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

### Project Path

The project root is added to the Python path so the notebook can import reusable project modules from `src`.


In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

### Project Imports

Reusable project utilities are imported from the `src` package.

In [3]:
from src.data import load_development_data

from src.features import (
    build_features,
    TARGET,
    YEAR,
    PRIMARY_FEATURES,
)

## Development Data

Only the 2020-2023 development data is used for model development and validation.

The raw development data is loaded and the deterministic feature-engineering transformations defined in `src/features.py` are applied before constructing the modeling dataset.

The 2024 final holdout set is not loaded at this stage.

In [4]:
train_df = load_development_data()
features_df = build_features(train_df)

### Validation

The loaded dataset is validated before modeling to ensure that only the intended development years are present and that the feature-engineering pipeline preserves the number of observations.

In [5]:
print("Raw shape:", train_df.shape)
print("Feature-engineered shape:", features_df.shape)
print("Years:", sorted(features_df[YEAR].unique()))

assert set(features_df[YEAR].unique()) == {
    2020, 2021, 2022, 2023
}

assert len(features_df) == len(train_df)

Raw shape: (41626, 46)
Feature-engineered shape: (41626, 63)
Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


## Modeling Dataset

The modeling dataset is constructed from the primary feature set defined during feature engineering.

The target variable is separated from the predictors, while the accident year is retained only for defining temporal cross-validation folds. The year itself is not provided to the model as a predictor.

Keeping the temporal variable separate ensures that it can be used to control the evaluation process without allowing the model to learn directly from the accident year.

### Target Encoding

The original severity codes are remapped from `1, 2, 3` to `0, 1, 2` for compatibility across all classifiers.

The class meaning remains unchanged:
- `0`: Fatal
- `1`: Serious
- `2`: Slight



In [6]:
X = features_df[PRIMARY_FEATURES].copy()

TARGET_MAPPING = {
    1: 0,  # Fatal
    2: 1,  # Serious
    3: 2,  # Slight
}

y = (
    features_df[TARGET]
    .map(TARGET_MAPPING)
    .astype(int)
)

years = features_df[YEAR].copy()

### Validation

The modeling inputs are checked to confirm that the target and temporal split variable are excluded from the predictors and that all rows remain aligned.

In [7]:
print("X shape:", X.shape)
print("Target shape:", y.shape)
print("Number of primary features:", len(PRIMARY_FEATURES))

X shape: (41626, 28)
Target shape: (41626,)
Number of primary features: 28


In [8]:
assert TARGET not in X.columns
assert YEAR not in X.columns

assert len(X) == len(y)
assert len(X) == len(years)

assert X.index.equals(y.index)
assert X.index.equals(years.index)

### Categorical Representation

The primary predictors are initially treated as categorical variables.

Many source variables are stored as numeric codes, but their numeric values represent categories rather than continuous quantities. For example, `SUG_DEREH = 1, 2, 3, 4` represents different road types and does not imply meaningful numeric distances between the values.

All primary predictors are therefore converted to a consistent string representation before categorical preprocessing.

Missing values are preserved at this stage and are handled later according to each model's preprocessing strategy.

In [9]:
def prepare_categorical_features(
    df: pd.DataFrame,
) -> pd.DataFrame:
    result = df.copy()

    for column in result.columns:
        result[column] = result[column].map(
            lambda value: str(value)
            if pd.notna(value)
            else np.nan
        )

    return result

In [10]:
X = prepare_categorical_features(X)

### Type Validation

The categorical representation is checked to ensure that each non-missing predictor value has a consistent string type before entering the preprocessing pipeline.

In [11]:
for column in X.columns:
    types = (
        X[column]
        .dropna()
        .map(type)
        .unique()
    )

    assert len(types) <= 1, (
        f"{column} contains mixed types: {types}"
    )

In [12]:
pd.DataFrame({
    "dtype": X.dtypes,
    "sample_type": [
        type(X[column].dropna().iloc[0]).__name__
        if X[column].notna().any()
        else None
        for column in X.columns
    ],
})

,dtype,sample_type
SUG_DEREH,str,str
SUG_TEUNA,str,str
MEHIRUT_MUTERET,str,str
ROHAV,str,str
SUG_EZEM,str,str
HODESH_TEUNA,str,str
YOM_BASHAVUA,str,str
YEHIDA,str,str
MAHOZ,str,str
NAFA,str,str


## Temporal Cross-Validation

Model evaluation uses expanding-window temporal cross-validation to reflect the intended prediction setting.

Each validation fold contains a later year than the years used for training:

| Fold | Training years | Validation year |
| --- | --- | --- |
| 1 | 2020 | 2021 |
| 2 | 2020-2021 | 2022 |
| 3 | 2020-2022 | 2023 |

This preserves chronological order and prevents the model from being trained on observations that occur after the validation period.

An expanding window is used so that each successive fold includes all historical data that would have been available at that point in time.

### Fold Definition

The temporal folds are defined explicitly rather than generated by a generic cross-validation splitter.

This makes the evaluation periods transparent and ensures that the intended expanding-window structure is preserved.

In [13]:
TEMPORAL_FOLDS = [
    {
        "name": "2020 -> 2021",
        "train_years": [2020],
        "validation_year": 2021,
    },
    {
        "name": "2020-2021 -> 2022",
        "train_years": [2020, 2021],
        "validation_year": 2022,
    },
    {
        "name": "2020-2022 -> 2023",
        "train_years": [2020, 2021, 2022],
        "validation_year": 2023,
    },
]

### Fold Sizes

The number of training and validation observations is inspected for each fold to verify that the temporal splits were constructed as intended.

As the training window expands, the amount of historical training data increases while each validation set contains one complete future year.

In [14]:
fold_summary = []

for fold in TEMPORAL_FOLDS:
    train_mask = years.isin(
        fold["train_years"]
    )

    validation_mask = years.eq(
        fold["validation_year"]
    )

    fold_summary.append({
        "fold": fold["name"],
        "train_rows": train_mask.sum(),
        "validation_rows": validation_mask.sum(),
    })

fold_summary = pd.DataFrame(
    fold_summary
)

fold_summary

,fold,train_rows,validation_rows
0,2020 -> 2021,10836,11554
1,2020-2021 -> 2022,22390,10404
2,2020-2022 -> 2023,32794,8832


### Fold Validation

Each temporal fold is validated to ensure that training observations always precede the validation year and that the training and validation sets do not overlap.

In [15]:
for fold in TEMPORAL_FOLDS:
    train_mask = years.isin(
        fold["train_years"]
    )

    validation_mask = years.eq(
        fold["validation_year"]
    )

    assert not (
        train_mask & validation_mask
    ).any()

    assert max(
        fold["train_years"]
    ) < fold["validation_year"]

## Evaluation Metrics

Accident severity is an imbalanced three-class classification problem, with the `Slight` class substantially more common than the `Serious` and `Fatal` classes.

Because of this imbalance, no single metric provides a complete view of model performance. Each model is therefore evaluated using several complementary metrics.

### Evaluation Framework

The following metrics are used throughout model evaluation:

| Metric | Purpose |
| --- | --- |
| **Accuracy** | Measures the overall proportion of correct predictions. It is easy to interpret, but can be misleading when one class dominates the dataset. |
| **Macro F1** | Calculates F1 independently for each severity class and averages the three scores equally. This is the primary model-selection metric because it gives equal importance to Fatal, Serious, and Slight accidents. |
| **Weighted F1** | Calculates F1 for each class and averages the results according to class frequency. It provides an overall performance measure while reflecting the observed class distribution. |
| **Class Recall** | Measures the proportion of actual observations from each severity class that the model correctly identifies. Recall is reported separately for Fatal, Serious, and Slight accidents. |

For more detailed analysis of selected models, additional class-level metrics are examined:

- **Precision**, to measure how often predictions for a given class are correct.
- **F1 score per class**, to balance precision and recall for each severity level.
- **Confusion matrices**, to show which severity classes are most frequently confused with one another.

### Primary Metric

**Macro F1** is used as the primary metric for comparing model configurations.

Unlike Accuracy or Weighted F1, Macro F1 gives each severity class equal influence on the final score, regardless of how frequently that class appears in the dataset.

This is particularly important for the Fatal class, which represents only a small portion of the observations. A model that predicts the majority `Slight` class well but performs poorly on Fatal and Serious accidents should therefore receive a relatively low primary score.

Model selection will consider the mean Macro F1 across all temporal validation folds. Fold-level results, temporal stability, and class-specific recall will also be inspected before selecting a final model configuration.

### Class Definitions

The encoded target uses the following class definitions:

| Code | Severity |
| ---: | --- |
| 0 | Fatal |
| 1 | Serious |
| 2 | Slight |

The class order is defined explicitly so that class-specific metrics are reported consistently across all models and folds.

In [16]:
CLASS_LABELS = [0, 1, 2]

CLASS_NAMES = {
    0: "Fatal",
    1: "Serious",
    2: "Slight",
}

### Metric Calculation

A reusable evaluation function calculates the same set of metrics for every model and temporal fold.

Using a single function ensures that all models are evaluated consistently and prevents differences in metric definitions between experiments.

In [17]:
def calculate_metrics(y_true, y_pred) -> dict:
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=CLASS_LABELS,
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=CLASS_LABELS,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            labels=CLASS_LABELS,
            average="weighted",
            zero_division=0,
        ),

        "fatal_precision": precision[0],
        "fatal_recall": recall[0],
        "fatal_f1": f1[0],

        "serious_precision": precision[1],
        "serious_recall": recall[1],
        "serious_f1": f1[1],

        "slight_precision": precision[2],
        "slight_recall": recall[2],
        "slight_f1": f1[2],
    }

## Modeling Helpers

Reusable functions used across all model experiments.

### Temporal Evaluation Helper

A reusable evaluation function trains and evaluates each model across the predefined temporal folds.

For each fold, it creates a fresh model instance, fits it only on historical training data, evaluates it on the following year, and stores both metrics and predictions for later analysis.

All models therefore use the same temporal evaluation procedure.

In [18]:
def evaluate_temporal_cv(
    model,
    model_name,
    X,
    y,
    years,
    fit_params=None,
    fit_params_factory=None,
):
    fold_results = []
    predictions = []

    for fold in TEMPORAL_FOLDS:
        train_mask = years.isin(
            fold["train_years"]
        )
        validation_mask = years.eq(
            fold["validation_year"]
        )

        X_train = X.loc[train_mask]
        y_train = y.loc[train_mask]

        X_validation = X.loc[
            validation_mask
        ]
        y_validation = y.loc[
            validation_mask
        ]

        fold_model = clone(model)

        current_fit_params = dict(
            fit_params or {}
        )

        if fit_params_factory is not None:
            current_fit_params.update(
                fit_params_factory(
                    y_train
                )
            )

        fold_model.fit(
            X_train,
            y_train,
            **current_fit_params,
        )

        y_pred = fold_model.predict(
            X_validation
        )

        y_pred = np.asarray(
            y_pred
        ).ravel()

        metrics = calculate_metrics(
            y_validation,
            y_pred,
        )

        fold_results.append({
            "model": model_name,
            "fold": fold["name"],
            "train_rows": len(X_train),
            "validation_rows": len(
                X_validation
            ),
            **metrics,
        })

        predictions.append(
            pd.DataFrame({
                "model": model_name,
                "fold": fold["name"],
                "year": fold[
                    "validation_year"
                ],
                "y_true":
                    y_validation.to_numpy(),
                "y_pred": y_pred,
            })
        )

    return (
        pd.DataFrame(fold_results),
        pd.concat(
            predictions,
            ignore_index=True,
        ),
    )

### Model Summary

Performance for each model is summarized using an equal-weight mean across the three temporal validation folds, so that every validation year contributes equally regardless of its number of observations.


In [19]:
SUMMARY_METRICS = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "fatal_recall",
    "serious_recall",
    "slight_recall",
]

def summarize_model_results(
    results: pd.DataFrame,
) -> pd.DataFrame:
    return (
        results[SUMMARY_METRICS]
        .mean()
        .to_frame("mean")
    )

### Model Comparison

A reusable helper summarizes the results of multiple model configurations evaluated with temporal cross-validation.

For each model, the comparison includes mean performance across validation folds together with the Macro F1 score from each individual fold.

Keeping both the average and fold-level results makes it possible to compare overall performance while also identifying models whose results vary substantially across validation years.

In [20]:
def build_model_comparison(
    *model_results: pd.DataFrame,
) -> pd.DataFrame:
    results = pd.concat(
        model_results,
        ignore_index=True,
    )

    mean_metrics = (
        results
        .groupby(
            "model",
            sort=False,
        )
        .agg(
            mean_macro_f1=("macro_f1", "mean"),
            mean_accuracy=("accuracy", "mean"),
            mean_weighted_f1=("weighted_f1", "mean"),
            mean_fatal_recall=("fatal_recall", "mean"),
            mean_serious_recall=("serious_recall", "mean"),
            mean_slight_recall=("slight_recall", "mean"),
        )
    )

    macro_f1_by_fold = (
        results
        .pivot(
            index="model",
            columns="fold",
            values="macro_f1",
        )
        .rename(
            columns={
                "2020 -> 2021": "macro_f1_2021",
                "2020-2021 -> 2022": "macro_f1_2022",
                "2020-2022 -> 2023": "macro_f1_2023",
            }
        )
    )

    comparison = mean_metrics.join(
        macro_f1_by_fold
    )

    column_order = [
        "mean_macro_f1",
        "macro_f1_2021",
        "macro_f1_2022",
        "macro_f1_2023",
        "mean_accuracy",
        "mean_weighted_f1",
        "mean_fatal_recall",
        "mean_serious_recall",
        "mean_slight_recall",
    ]

    return comparison[column_order]

## Preprocessing

### One-Hot Encoded Models

Logistic Regression, Random Forest, and XGBoost use missing-value imputation followed by one-hot encoding.

- **Missing values** are replaced with an explicit `Missing` category, preserving the distinction between unavailable and observed information.
- **One-hot encoding** (`handle_unknown="ignore"`) treats every primary feature as categorical, since the underlying codes (e.g. `SUG_DEREH`) represent category labels rather than continuous quantities. Categories not seen during training are ignored rather than causing an error.


In [21]:
categorical_imputer = SimpleImputer(
    strategy="constant",
    fill_value="Missing",
)

In [22]:
categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
)

In [23]:
categorical_preprocessor = Pipeline([
    (
        "imputer",
        categorical_imputer,
    ),
    (
        "onehot",
        categorical_encoder,
    ),
])

In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_preprocessor,
            PRIMARY_FEATURES,
        ),
    ],
    remainder="drop",
)

### Native Categorical Models

CatBoost handles categorical features natively via `cat_features`, so the one-hot encoding pipeline above is not applied. The same primary feature set is used, with missing values represented explicitly as `Missing`. The result is validated to confirm no missing values remain and that rows stay aligned with `X`.


In [25]:
X_catboost = X.fillna(
    "Missing"
).copy()

In [26]:
assert X_catboost.notna().all().all()
assert len(X_catboost) == len(X)
assert X_catboost.index.equals(X.index)


## Models

Each model is evaluated using the same temporal cross-validation procedure, primary feature set, and evaluation metrics defined above.


### Dummy Baseline

A majority-class dummy classifier serves as a no-skill reference baseline, always predicting the most frequent severity class observed in each training fold.

It provides a minimum reference point that more meaningful models should outperform, particularly on Macro F1 and minority-class recall.


#### Model Definition

The dummy classifier uses the `most_frequent` strategy, predicting the majority class observed in the training portion of each temporal fold.

In [27]:
dummy_model = DummyClassifier(
    strategy="most_frequent"
)

In [28]:
dummy_results, dummy_predictions = (
    evaluate_temporal_cv(
        model=dummy_model,
        model_name="Dummy",
        X=X,
        y=y,
        years=years,
    )
)

In [29]:
dummy_results

,model,fold,train_rows,validation_rows,accuracy,macro_f1,weighted_f1,fatal_precision,fatal_recall,fatal_f1,serious_precision,serious_recall,serious_f1,slight_precision,slight_recall,slight_f1
0,Dummy,2020 -> 2021,10836,11554,0.779817,0.292096,0.683344,0.0,0.0,0.0,0.0,0.0,0.0,0.779817,1.0,0.876289
1,Dummy,2020-2021 -> 2022,22390,10404,0.746348,0.284918,0.637943,0.0,0.0,0.0,0.0,0.0,0.0,0.746348,1.0,0.854753
2,Dummy,2020-2022 -> 2023,32794,8832,0.692935,0.272873,0.567250,0.0,0.0,0.0,0.0,0.0,0.0,0.692935,1.0,0.818620


#### Baseline Summary


In [30]:
dummy_summary = summarize_model_results(
    dummy_results
)

dummy_summary

,mean
accuracy,0.739700
macro_f1,0.283296
weighted_f1,0.629512
fatal_recall,0.000000
serious_recall,0.000000
slight_recall,1.000000


#### Interpretation

The dummy classifier reaches a mean Macro F1 of only 0.283, with zero recall for the Fatal and Serious classes despite a relatively high accuracy of 0.740 from always predicting the majority `Slight` class.

This confirms that Accuracy and frequency-weighted metrics are unreliable on their own for this imbalanced problem, and establishes the no-skill baseline for all subsequent models.


### Logistic Regression

Logistic Regression provides a simple learned baseline using the shared one-hot encoded feature representation.

No class weighting or hyperparameter tuning is applied.


#### Modeling Pipeline


#### Solver and Iteration Limit

- `solver="lbfgs"`: suitable for the multiclass Logistic Regression baseline.
- `max_iter=3000`: provides sufficient iterations for convergence with the one-hot encoded feature space.

In [31]:
logistic_model = Pipeline([
    (
        "preprocessor",
        preprocessor,
    ),
    (
        "classifier",
        LogisticRegression(
            solver="lbfgs",
            max_iter=3000,
        ),
    ),
])

#### Temporal Evaluation


In [32]:
logistic_results, logistic_predictions = (
    evaluate_temporal_cv(
        model=logistic_model,
        model_name="Logistic Regression",
        X=X,
        y=y,
        years=years,
    )
)

logistic_results

,model,fold,train_rows,validation_rows,accuracy,macro_f1,weighted_f1,fatal_precision,fatal_recall,fatal_f1,serious_precision,serious_recall,serious_f1,slight_precision,slight_recall,slight_f1
0,Logistic Regression,2020 -> 2021,10836,11554,0.809157,0.435237,0.763295,0.413793,0.035714,0.065753,0.710709,0.231431,0.349163,0.816768,0.979578,0.890795
1,Logistic Regression,2020-2021 -> 2022,22390,10404,0.789216,0.444817,0.745640,0.346154,0.028213,0.052174,0.728587,0.282328,0.406959,0.796181,0.971925,0.875319
2,Logistic Regression,2020-2022 -> 2023,32794,8832,0.767437,0.464100,0.723438,0.440000,0.032544,0.060606,0.809524,0.336563,0.475454,0.763171,0.975163,0.856241


#### Mean Performance


In [33]:
logistic_summary = summarize_model_results(logistic_results)

logistic_summary

,mean
accuracy,0.788603
macro_f1,0.448052
weighted_f1,0.744124
fatal_recall,0.032157
serious_recall,0.283441
slight_recall,0.975556


#### Interpretation

Logistic Regression raises mean Macro F1 from 0.283 (dummy) to 0.448, but remains strongly biased toward the majority class: Slight recall reaches 97.6% while Fatal recall is only 3.2%.

This establishes a simple learned reference point before introducing class-imbalance handling.


### Balanced Logistic Regression

Balanced Logistic Regression tests whether the standard baseline's bias toward the majority class is driven by class imbalance, by adding automatic class weighting.

All other pipeline components are unchanged, isolating the effect of class weighting for direct comparison with the standard baseline.


#### Class Weighting

`class_weight="balanced"` increases the contribution of minority classes based on their frequency in each training fold.

This allows the effect of imbalance handling to be compared directly with the standard Logistic Regression baseline.


#### Model Definition

The balanced model clones the standard Logistic Regression pipeline, changing only the classifier's `class_weight` parameter.


In [34]:
balanced_logistic_model = clone(
    logistic_model
)

_ = balanced_logistic_model.set_params(
    classifier__class_weight="balanced"
)

#### Temporal Evaluation


In [35]:
balanced_logistic_results, balanced_logistic_predictions = (
    evaluate_temporal_cv(
        model=balanced_logistic_model,
        model_name="Balanced Logistic Regression",
        X=X,
        y=y,
        years=years,
    )
)

balanced_logistic_results

,model,fold,train_rows,validation_rows,accuracy,macro_f1,weighted_f1,fatal_precision,fatal_recall,fatal_f1,serious_precision,serious_recall,serious_f1,slight_precision,slight_recall,slight_f1
0,Balanced Logistic Regression,2020 -> 2021,10836,11554,0.606370,0.433278,0.665812,0.077251,0.538690,0.135125,0.383139,0.461051,0.418499,0.886024,0.644506,0.746209
1,Balanced Logistic Regression,2020-2021 -> 2022,22390,10404,0.612937,0.453617,0.669027,0.085626,0.605016,0.150019,0.457488,0.456897,0.457192,0.878450,0.659884,0.753640
2,Balanced Logistic Regression,2020-2022 -> 2023,32794,8832,0.616735,0.476545,0.667834,0.095989,0.594675,0.165296,0.582237,0.447346,0.505955,0.851445,0.683660,0.758383


#### Mean Performance


In [36]:
balanced_logistic_summary = summarize_model_results(balanced_logistic_results)

balanced_logistic_summary

,mean
accuracy,0.612014
macro_f1,0.454480
weighted_f1,0.667558
fatal_recall,0.579460
serious_recall,0.455098
slight_recall,0.662683


#### Comparison with Standard Logistic Regression

Mean performance of the standard and balanced configurations is compared directly, isolating the effect of class weighting.


In [37]:
logistic_comparison = pd.DataFrame({
    "Logistic Regression":
        logistic_summary["mean"],
    "Balanced Logistic Regression":
        balanced_logistic_summary["mean"],
})

logistic_comparison

,Logistic Regression,Balanced Logistic Regression
accuracy,0.788603,0.612014
macro_f1,0.448052,0.454480
weighted_f1,0.744124,0.667558
fatal_recall,0.032157,0.579460
serious_recall,0.283441,0.455098
slight_recall,0.975556,0.662683


#### Interpretation

Class weighting sharply improves minority-class recall (Fatal: 0.032 → 0.579; Serious: 0.283 → 0.455) at the cost of Slight recall (0.976 → 0.663), though mean Macro F1 improves only marginally to 0.454.

The balanced configuration is retained as a candidate for comparison with the remaining model families.


### Random Forest

Random Forest is evaluated as a non-linear tree-based baseline using the shared one-hot preprocessing pipeline.

No class weighting or hyperparameter tuning is applied in this initial configuration.


#### Model Definition

A Random Forest with `n_estimators=300` and a fixed random state for reproducibility.


In [38]:
random_forest_model = Pipeline([
    (
        "preprocessor",
        preprocessor,
    ),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

#### Temporal Evaluation


In [39]:
random_forest_results, random_forest_predictions = (
    evaluate_temporal_cv(
        model=random_forest_model,
        model_name="Random Forest",
        X=X,
        y=y,
        years=years,
    )
)

random_forest_results

,model,fold,train_rows,validation_rows,accuracy,macro_f1,weighted_f1,fatal_precision,fatal_recall,fatal_f1,serious_precision,serious_recall,serious_f1,slight_precision,slight_recall,slight_f1
0,Random Forest,2020 -> 2021,10836,11554,0.809676,0.425733,0.763090,0.500000,0.017857,0.034483,0.709766,0.233696,0.351618,0.816736,0.980355,0.891097
1,Random Forest,2020-2021 -> 2022,22390,10404,0.792195,0.451013,0.749636,0.400000,0.031348,0.058140,0.737186,0.291379,0.417671,0.798563,0.973084,0.877228
2,Random Forest,2020-2022 -> 2023,32794,8832,0.767097,0.457277,0.721830,0.347826,0.023669,0.044321,0.815353,0.331087,0.470941,0.762396,0.977288,0.856570


#### Mean Performance


In [40]:
random_forest_summary = summarize_model_results(random_forest_results)

random_forest_summary

,mean
accuracy,0.789656
macro_f1,0.444674
weighted_f1,0.744852
fatal_recall,0.024291
serious_recall,0.285387
slight_recall,0.976909


#### Comparison with Previous Models

Random Forest is added to the current model-screening comparison.


In [41]:
current_model_comparison = build_model_comparison(
    dummy_results,
    logistic_results,
    balanced_logistic_results,
    random_forest_results,
)

current_model_comparison.round(3)

,mean_macro_f1,macro_f1_2021,macro_f1_2022,macro_f1_2023,mean_accuracy,mean_weighted_f1,mean_fatal_recall,mean_serious_recall,mean_slight_recall
model,,,,,,,,,
Dummy,0.283,0.292,0.285,0.273,0.740,0.630,0.000,0.000,1.000
Logistic Regression,0.448,0.435,0.445,0.464,0.789,0.744,0.032,0.283,0.976
Balanced Logistic Regression,0.454,0.433,0.454,0.477,0.612,0.668,0.579,0.455,0.663
Random Forest,0.445,0.426,0.451,0.457,0.790,0.745,0.024,0.285,0.977


#### Interpretation

All learned models clearly outperform the dummy baseline, though standard Logistic Regression and Random Forest perform similarly (mean Macro F1 0.448 vs. 0.445) with limited Fatal recall.

Balanced Logistic Regression currently leads on Macro F1 (0.454), trading Slight recall for better minority-class detection. Differences remain small enough that further model families should be evaluated before shortlisting.


### Balanced Random Forest

Balanced Random Forest adds class weighting to the standard Random Forest baseline to test whether it improves minority-class detection for a non-linear model, as it did for Logistic Regression.

All other components (features, preprocessing, tree configuration) remain unchanged.


#### Model Definition

Clones the standard Random Forest pipeline, changing only the classifier's `class_weight` parameter.


In [42]:
balanced_random_forest_model = clone(
    random_forest_model
)

_ = balanced_random_forest_model.set_params(
    classifier__class_weight="balanced"
)

#### Temporal Evaluation


In [43]:
balanced_random_forest_results, balanced_random_forest_predictions = (
    evaluate_temporal_cv(
        model=balanced_random_forest_model,
        model_name="Balanced Random Forest",
        X=X,
        y=y,
        years=years,
    )
)

balanced_random_forest_results

,model,fold,train_rows,validation_rows,accuracy,macro_f1,weighted_f1,fatal_precision,fatal_recall,fatal_f1,serious_precision,serious_recall,serious_f1,slight_precision,slight_recall,slight_f1
0,Balanced Random Forest,2020 -> 2021,10836,11554,0.780076,0.481263,0.767444,0.238095,0.089286,0.129870,0.480678,0.411232,0.443251,0.846525,0.896226,0.870667
1,Balanced Random Forest,2020-2021 -> 2022,22390,10404,0.767397,0.495631,0.753748,0.253968,0.100313,0.143820,0.532091,0.443103,0.483537,0.829619,0.891693,0.859537
2,Balanced Random Forest,2020-2022 -> 2023,32794,8832,0.754755,0.504664,0.736472,0.264151,0.082840,0.126126,0.617536,0.483572,0.542405,0.799476,0.897059,0.845461


#### Mean Performance


In [44]:
balanced_random_forest_summary = summarize_model_results(balanced_random_forest_results)

balanced_random_forest_summary

,mean
accuracy,0.767410
macro_f1,0.493853
weighted_f1,0.752554
fatal_recall,0.090813
serious_recall,0.445969
slight_recall,0.894993


#### Comparison with Previous Models

Balanced Random Forest is added to the current model-screening comparison.


In [45]:
current_model_comparison = build_model_comparison(
    dummy_results,
    logistic_results,
    balanced_logistic_results,
    random_forest_results,
    balanced_random_forest_results,
)

current_model_comparison.round(3)

,mean_macro_f1,macro_f1_2021,macro_f1_2022,macro_f1_2023,mean_accuracy,mean_weighted_f1,mean_fatal_recall,mean_serious_recall,mean_slight_recall
model,,,,,,,,,
Dummy,0.283,0.292,0.285,0.273,0.740,0.630,0.000,0.000,1.000
Logistic Regression,0.448,0.435,0.445,0.464,0.789,0.744,0.032,0.283,0.976
Balanced Logistic Regression,0.454,0.433,0.454,0.477,0.612,0.668,0.579,0.455,0.663
Random Forest,0.445,0.426,0.451,0.457,0.790,0.745,0.024,0.285,0.977
Balanced Random Forest,0.494,0.481,0.496,0.505,0.767,0.753,0.091,0.446,0.895


#### Interpretation

Balanced Random Forest raises mean Macro F1 from 0.445 to 0.494, the strongest result so far, with consistent gains across folds and improved Fatal/Serious recall (0.024 → 0.091, 0.285 → 0.446) while retaining high Slight recall (0.895).

It is retained as a strong candidate alongside Balanced Logistic Regression.


### CatBoost

CatBoost is evaluated as the first boosting model family. Unlike Logistic Regression and Random Forest, it handles categorical features natively via `cat_features`, without requiring one-hot encoding (see Preprocessing).


#### Model Definition

Initial CatBoost configuration:

- `iterations=500`: provides a reasonable boosting baseline without tuning.
- `loss_function="MultiClass"`: matches the three-class severity prediction problem.


In [46]:
catboost_model = CatBoostClassifier(
    iterations=500,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

#### Temporal Evaluation

In [47]:
catboost_results, catboost_predictions = (
    evaluate_temporal_cv(
        model=catboost_model,
        model_name="CatBoost",
        X=X_catboost,
        y=y,
        years=years,
        fit_params={
            "cat_features": list(X_catboost.columns),
        },
    )
)

catboost_results

,model,fold,train_rows,validation_rows,accuracy,macro_f1,weighted_f1,fatal_precision,fatal_recall,fatal_f1,serious_precision,serious_recall,serious_f1,slight_precision,slight_recall,slight_f1
0,CatBoost,2020 -> 2021,10836,11554,0.814437,0.448681,0.772585,0.382353,0.038690,0.070270,0.709877,0.260417,0.381047,0.823716,0.979134,0.894726
1,CatBoost,2020-2021 -> 2022,22390,10404,0.794887,0.474349,0.756075,0.425532,0.062696,0.109290,0.738144,0.308621,0.435258,0.802599,0.970251,0.878498
2,CatBoost,2020-2022 -> 2023,32794,8832,0.770947,0.473376,0.727720,0.468750,0.044379,0.081081,0.811871,0.339933,0.479216,0.766974,0.978268,0.859831


#### Mean Performance


In [48]:
catboost_summary = summarize_model_results(catboost_results)

catboost_summary

,mean
accuracy,0.793423
macro_f1,0.465469
weighted_f1,0.752127
fatal_recall,0.048588
serious_recall,0.302990
slight_recall,0.975884


#### Comparison with Previous Models

CatBoost is added to the current model-screening comparison.


In [49]:
current_model_comparison = build_model_comparison(
    dummy_results,
    logistic_results,
    balanced_logistic_results,
    random_forest_results,
    balanced_random_forest_results,
    catboost_results,
)

current_model_comparison.round(3)

,mean_macro_f1,macro_f1_2021,macro_f1_2022,macro_f1_2023,mean_accuracy,mean_weighted_f1,mean_fatal_recall,mean_serious_recall,mean_slight_recall
model,,,,,,,,,
Dummy,0.283,0.292,0.285,0.273,0.740,0.630,0.000,0.000,1.000
Logistic Regression,0.448,0.435,0.445,0.464,0.789,0.744,0.032,0.283,0.976
Balanced Logistic Regression,0.454,0.433,0.454,0.477,0.612,0.668,0.579,0.455,0.663
Random Forest,0.445,0.426,0.451,0.457,0.790,0.745,0.024,0.285,0.977
Balanced Random Forest,0.494,0.481,0.496,0.505,0.767,0.753,0.091,0.446,0.895
CatBoost,0.465,0.449,0.474,0.473,0.793,0.752,0.049,0.303,0.976


#### Interpretation

CatBoost reaches a mean Macro F1 of 0.465, ahead of the standard Logistic Regression and Random Forest baselines, but remains biased toward the Slight class with Fatal recall of only 0.049.

Balanced Random Forest (0.494) remains the strongest configuration so far.


### Balanced CatBoost

Automatic class weighting is added to the CatBoost baseline to evaluate whether minority-class detection can be improved.

#### Model Definition

Same configuration as the CatBoost baseline, with `auto_class_weights="Balanced"` added to increase the influence of less frequent classes during training.


In [50]:
balanced_catboost_model = CatBoostClassifier(
    iterations=500,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

#### Temporal Evaluation

In [51]:
balanced_catboost_results, balanced_catboost_predictions = (
    evaluate_temporal_cv(
        model=balanced_catboost_model,
        model_name="Balanced CatBoost",
        X=X_catboost,
        y=y,
        years=years,
        fit_params={
            "cat_features": list(X_catboost.columns),
        },
    )
)

balanced_catboost_results

,model,fold,train_rows,validation_rows,accuracy,macro_f1,weighted_f1,fatal_precision,fatal_recall,fatal_f1,serious_precision,serious_recall,serious_f1,slight_precision,slight_recall,slight_f1
0,Balanced CatBoost,2020 -> 2021,10836,11554,0.731175,0.485964,0.741514,0.158590,0.214286,0.182278,0.404971,0.479620,0.439146,0.862345,0.812098,0.836468
1,Balanced CatBoost,2020-2021 -> 2022,22390,10404,0.713476,0.504884,0.731130,0.132576,0.329154,0.189019,0.483044,0.528017,0.504530,0.861080,0.784675,0.821104
2,Balanced CatBoost,2020-2022 -> 2023,32794,8832,0.698370,0.515829,0.716147,0.131552,0.378698,0.195271,0.575192,0.505897,0.538324,0.838503,0.790686,0.813893


#### Mean Performance

In [52]:
balanced_catboost_summary = summarize_model_results(balanced_catboost_results)

balanced_catboost_summary

,mean
accuracy,0.714340
macro_f1,0.502226
weighted_f1,0.729597
fatal_recall,0.307379
serious_recall,0.504511
slight_recall,0.795820


#### Comparison with CatBoost

The balanced configuration is compared directly with the standard CatBoost baseline to isolate the effect of class weighting.

In [53]:
catboost_comparison = pd.DataFrame({
    "CatBoost":
        catboost_summary["mean"],
    "Balanced CatBoost":
        balanced_catboost_summary["mean"],
})

catboost_comparison.round(3)

,CatBoost,Balanced CatBoost
accuracy,0.793,0.714
macro_f1,0.465,0.502
weighted_f1,0.752,0.730
fatal_recall,0.049,0.307
serious_recall,0.303,0.505
slight_recall,0.976,0.796


#### Comparison with Previous Models

Balanced CatBoost is added to the model-screening comparison.

In [54]:
current_model_comparison = build_model_comparison(
    dummy_results,
    logistic_results,
    balanced_logistic_results,
    random_forest_results,
    balanced_random_forest_results,
    catboost_results,
    balanced_catboost_results,
)

current_model_comparison.round(3)

,mean_macro_f1,macro_f1_2021,macro_f1_2022,macro_f1_2023,mean_accuracy,mean_weighted_f1,mean_fatal_recall,mean_serious_recall,mean_slight_recall
model,,,,,,,,,
Dummy,0.283,0.292,0.285,0.273,0.740,0.630,0.000,0.000,1.000
Logistic Regression,0.448,0.435,0.445,0.464,0.789,0.744,0.032,0.283,0.976
Balanced Logistic Regression,0.454,0.433,0.454,0.477,0.612,0.668,0.579,0.455,0.663
Random Forest,0.445,0.426,0.451,0.457,0.790,0.745,0.024,0.285,0.977
Balanced Random Forest,0.494,0.481,0.496,0.505,0.767,0.753,0.091,0.446,0.895
CatBoost,0.465,0.449,0.474,0.473,0.793,0.752,0.049,0.303,0.976
Balanced CatBoost,0.502,0.486,0.505,0.516,0.714,0.730,0.307,0.505,0.796


#### Interpretation

Balanced CatBoost achieves the highest mean Macro F1 so far, improving from 0.465 to 0.502 with consistent gains across folds and substantially better Fatal and Serious recall (0.049 → 0.307, 0.303 → 0.505).

It becomes the leading candidate, alongside Balanced Random Forest, for further analysis.


### XGBoost

XGBoost is evaluated as an additional boosting model using the same one-hot encoded primary feature set as the Logistic Regression and Random Forest models.

#### Model Definition

Initial XGBoost configuration:

- `n_estimators=500`: provides a reasonable boosting baseline without tuning.
- `learning_rate=0.1`: moderate learning rate suitable for an initial experiment.
- `max_depth=6`: allows non-linear interactions without starting with very deep trees.
- `objective="multi:softprob"`: configures multiclass classification.


In [55]:
xgboost_model = Pipeline([
    (
        "preprocessor",
        preprocessor,
    ),
    (
        "classifier",
        XGBClassifier(
            n_estimators=500,
            learning_rate=0.1,
            max_depth=6,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

#### Temporal Evaluation

In [56]:
xgboost_results, xgboost_predictions = (
    evaluate_temporal_cv(
        model=xgboost_model,
        model_name="XGBoost",
        X=X,
        y=y,
        years=years,
    )
)

xgboost_results

,model,fold,train_rows,validation_rows,accuracy,macro_f1,weighted_f1,fatal_precision,fatal_recall,fatal_f1,serious_precision,serious_recall,serious_f1,slight_precision,slight_recall,slight_f1
0,XGBoost,2020 -> 2021,10836,11554,0.801627,0.452636,0.764932,0.363636,0.059524,0.102302,0.593373,0.267663,0.368914,0.823669,0.960155,0.886691
1,XGBoost,2020-2021 -> 2022,22390,10404,0.788639,0.471692,0.753031,0.295775,0.065831,0.107692,0.680259,0.316379,0.431892,0.805057,0.959433,0.875492
2,XGBoost,2020-2022 -> 2023,32794,8832,0.765059,0.486013,0.725621,0.382353,0.076923,0.128079,0.763109,0.343302,0.473562,0.768711,0.966667,0.856398


#### Mean Performance

In [57]:
xgboost_summary = summarize_model_results(xgboost_results)

xgboost_summary

,mean
accuracy,0.785108
macro_f1,0.470114
weighted_f1,0.747861
fatal_recall,0.067426
serious_recall,0.309115
slight_recall,0.962085


#### Comparison with Previous Models

XGBoost is added to the current model-screening comparison.

In [58]:
current_model_comparison = build_model_comparison(
    dummy_results,
    logistic_results,
    balanced_logistic_results,
    random_forest_results,
    balanced_random_forest_results,
    catboost_results,
    balanced_catboost_results,
    xgboost_results,
)

current_model_comparison.round(3)

,mean_macro_f1,macro_f1_2021,macro_f1_2022,macro_f1_2023,mean_accuracy,mean_weighted_f1,mean_fatal_recall,mean_serious_recall,mean_slight_recall
model,,,,,,,,,
Dummy,0.283,0.292,0.285,0.273,0.740,0.630,0.000,0.000,1.000
Logistic Regression,0.448,0.435,0.445,0.464,0.789,0.744,0.032,0.283,0.976
Balanced Logistic Regression,0.454,0.433,0.454,0.477,0.612,0.668,0.579,0.455,0.663
Random Forest,0.445,0.426,0.451,0.457,0.790,0.745,0.024,0.285,0.977
Balanced Random Forest,0.494,0.481,0.496,0.505,0.767,0.753,0.091,0.446,0.895
CatBoost,0.465,0.449,0.474,0.473,0.793,0.752,0.049,0.303,0.976
Balanced CatBoost,0.502,0.486,0.505,0.516,0.714,0.730,0.307,0.505,0.796
XGBoost,0.470,0.453,0.472,0.486,0.785,0.748,0.067,0.309,0.962


#### Interpretation

XGBoost reaches a mean Macro F1 of 0.470, slightly ahead of the other unbalanced models, but still shows low Fatal recall (0.067).

Balanced CatBoost and Balanced Random Forest remain the strongest configurations evaluated so far.


### Balanced XGBoost

Class-balanced sample weights are added to the XGBoost baseline to improve minority-class detection.

#### Model Definition

The XGBoost configuration remains unchanged. Only class-balanced training weights are added.

In [59]:
balanced_xgboost_model = clone(
    xgboost_model
)

#### Class Weighting

Per-sample weights from `compute_sample_weight` give greater importance to minority-class observations. The weights are calculated independently within each training fold and provide XGBoost's equivalent of `class_weight="balanced"`.

In [60]:
def build_balanced_xgboost_fit_params(
    y_train,
):
    sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    return {
        "classifier__sample_weight":
            sample_weights,
    }

#### Temporal Evaluation

In [61]:
balanced_xgboost_results, balanced_xgboost_predictions = (
    evaluate_temporal_cv(
        model=balanced_xgboost_model,
        model_name="Balanced XGBoost",
        X=X,
        y=y,
        years=years,
        fit_params_factory=
            build_balanced_xgboost_fit_params,
    )
)

balanced_xgboost_results

,model,fold,train_rows,validation_rows,accuracy,macro_f1,weighted_f1,fatal_precision,fatal_recall,fatal_f1,serious_precision,serious_recall,serious_f1,slight_precision,slight_recall,slight_f1
0,Balanced XGBoost,2020 -> 2021,10836,11554,0.726155,0.475257,0.734292,0.157044,0.202381,0.176853,0.391061,0.443841,0.415783,0.852234,0.814872,0.833135
1,Balanced XGBoost,2020-2021 -> 2022,22390,10404,0.705882,0.482140,0.721166,0.109306,0.231975,0.148594,0.459311,0.506034,0.481542,0.850091,0.785061,0.816283
2,Balanced XGBoost,2020-2022 -> 2023,32794,8832,0.700521,0.516198,0.714375,0.141330,0.352071,0.201695,0.562646,0.508846,0.534395,0.831765,0.794118,0.812505


#### Mean Performance

In [62]:
balanced_xgboost_summary = summarize_model_results(balanced_xgboost_results)

balanced_xgboost_summary

,mean
accuracy,0.710853
macro_f1,0.491198
weighted_f1,0.723278
fatal_recall,0.262142
serious_recall,0.486240
slight_recall,0.798017


#### Comparison with XGBoost

The balanced configuration is compared directly with the standard XGBoost baseline.

In [63]:
xgboost_comparison = pd.DataFrame({
    "XGBoost":
        xgboost_summary["mean"],
    "Balanced XGBoost":
        balanced_xgboost_summary["mean"],
})

xgboost_comparison.round(3)

,XGBoost,Balanced XGBoost
accuracy,0.785,0.711
macro_f1,0.470,0.491
weighted_f1,0.748,0.723
fatal_recall,0.067,0.262
serious_recall,0.309,0.486
slight_recall,0.962,0.798


#### Comparison with Previous Models

Balanced XGBoost is added to the model-screening comparison.

In [64]:
current_model_comparison = build_model_comparison(
    dummy_results,
    logistic_results,
    balanced_logistic_results,
    random_forest_results,
    balanced_random_forest_results,
    catboost_results,
    balanced_catboost_results,
    xgboost_results,
    balanced_xgboost_results,
)

current_model_comparison.round(3)

,mean_macro_f1,macro_f1_2021,macro_f1_2022,macro_f1_2023,mean_accuracy,mean_weighted_f1,mean_fatal_recall,mean_serious_recall,mean_slight_recall
model,,,,,,,,,
Dummy,0.283,0.292,0.285,0.273,0.740,0.630,0.000,0.000,1.000
Logistic Regression,0.448,0.435,0.445,0.464,0.789,0.744,0.032,0.283,0.976
Balanced Logistic Regression,0.454,0.433,0.454,0.477,0.612,0.668,0.579,0.455,0.663
Random Forest,0.445,0.426,0.451,0.457,0.790,0.745,0.024,0.285,0.977
Balanced Random Forest,0.494,0.481,0.496,0.505,0.767,0.753,0.091,0.446,0.895
CatBoost,0.465,0.449,0.474,0.473,0.793,0.752,0.049,0.303,0.976
Balanced CatBoost,0.502,0.486,0.505,0.516,0.714,0.730,0.307,0.505,0.796
XGBoost,0.470,0.453,0.472,0.486,0.785,0.748,0.067,0.309,0.962
Balanced XGBoost,0.491,0.475,0.482,0.516,0.711,0.723,0.262,0.486,0.798


#### Interpretation

Balanced XGBoost improves mean Macro F1 from 0.470 to 0.491, with the expected trade-off of higher Fatal/Serious recall for lower Slight recall.

Balanced CatBoost remains the strongest configuration, with Balanced Random Forest and Balanced XGBoost as competitive alternatives.


## Model Screening Summary

The initial screening evaluated linear, bagging, and boosting model families with and without class-imbalance handling. Class weighting consistently improved minority-class recall across all model families.

Balanced CatBoost achieves the highest mean Macro F1 (0.502), followed by Balanced Random Forest (0.494) and Balanced XGBoost (0.491). These three configurations are retained for further analysis and tuning.


In [65]:
final_model_comparison = build_model_comparison(
    dummy_results,
    logistic_results,
    balanced_logistic_results,
    random_forest_results,
    balanced_random_forest_results,
    catboost_results,
    balanced_catboost_results,
    xgboost_results,
    balanced_xgboost_results,
)

final_model_comparison = (
    final_model_comparison
    .sort_values(
        "mean_macro_f1",
        ascending=False,
    )
)

final_model_comparison.round(3)


,mean_macro_f1,macro_f1_2021,macro_f1_2022,macro_f1_2023,mean_accuracy,mean_weighted_f1,mean_fatal_recall,mean_serious_recall,mean_slight_recall
model,,,,,,,,,
Balanced CatBoost,0.502,0.486,0.505,0.516,0.714,0.730,0.307,0.505,0.796
Balanced Random Forest,0.494,0.481,0.496,0.505,0.767,0.753,0.091,0.446,0.895
Balanced XGBoost,0.491,0.475,0.482,0.516,0.711,0.723,0.262,0.486,0.798
XGBoost,0.470,0.453,0.472,0.486,0.785,0.748,0.067,0.309,0.962
CatBoost,0.465,0.449,0.474,0.473,0.793,0.752,0.049,0.303,0.976
Balanced Logistic Regression,0.454,0.433,0.454,0.477,0.612,0.668,0.579,0.455,0.663
Logistic Regression,0.448,0.435,0.445,0.464,0.789,0.744,0.032,0.283,0.976
Random Forest,0.445,0.426,0.451,0.457,0.790,0.745,0.024,0.285,0.977
Dummy,0.283,0.292,0.285,0.273,0.740,0.630,0.000,0.000,1.000
